# Communicative-efficiency analysis

Reproduces the correlations and Figure 5 of Yin et al. (2024).

Inputs (see ``data/`` after running the pipeline scripts):
* ``data/scores/<split>/{finger_ind_all,listener_all,speaker_all,thumb}.json`` -- effort scores per FS split.
* ``data/freq/{nat,for,}asl_freq{,_type}.json`` -- ASL handshape frequencies.
* ``data/freq/wiki_char_{word,type}.json`` -- English letter frequencies.
* ``data/freq/wiki_ent_{word,type}.json`` -- English contextual confusability.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import json
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sns
from scipy import stats

from src.utils import remove_outliers

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']
plt.rcParams['font.size'] = 16

cmap = plt.cm.Set2
colors = cmap(np.linspace(0, 1, 8))

DATA_DIR = '../data'
PLOT_DIR = '../plots'
os.makedirs(PLOT_DIR, exist_ok=True)

ALPHABET = 'abcdefghiklmnopqrstuvwxy'  # 24 letters: J and Z require movement


## Load effort scores

We pool samples from all four corpus splits and report mean (after IQR outlier removal) per letter / letter pair.

In [ ]:
SPLITS = ['supp0_5', 'supp6_118', 'supp119_614', 'train0_22']

scores = {}
for split in SPLITS:
    s = {}
    for name in ['finger_ind_all', 'listener_all', 'speaker_all', 'thumb']:
        with open(f'{DATA_DIR}/scores/{split}/{name}.json') as f:
            s[name] = json.load(f)
    scores[split] = s


## Load usage statistics

In [ ]:
freq = {
    'en_word':       json.load(open(f'{DATA_DIR}/freq/wiki_char_word.json')),
    'en_type':       json.load(open(f'{DATA_DIR}/freq/wiki_char_type.json')),
    'conf_word':     json.load(open(f'{DATA_DIR}/freq/wiki_ent_word.json')),
    'conf_type':     json.load(open(f'{DATA_DIR}/freq/wiki_ent_type.json')),
    'asl_word':      json.load(open(f'{DATA_DIR}/freq/asl_freq.json')),
    'asl_type':      json.load(open(f'{DATA_DIR}/freq/asl_freq_type.json')),
    'nat_asl_word':  json.load(open(f'{DATA_DIR}/freq/nat_asl_freq.json')),
    'nat_asl_type':  json.load(open(f'{DATA_DIR}/freq/nat_asl_freq_type.json')),
    'for_asl_word':  json.load(open(f'{DATA_DIR}/freq/for_asl_freq.json')),
    'for_asl_type':  json.load(open(f'{DATA_DIR}/freq/for_asl_freq_type.json')),
}


## Aggregate per-letter and per-pair effort scores

Per the paper: finger independence is computed as the joint-pair score plus 2x the thumb effort, then averaged with IQR outlier removal across all four splits.

In [ ]:
data = defaultdict(list)

for letter in ALPHABET:
    data['char_label'].append(letter.upper())
    for k in ['en_word', 'en_type', 'asl_word', 'asl_type',
             'nat_asl_word', 'nat_asl_type', 'for_asl_word', 'for_asl_type']:
        data[k].append(freq[k].get(letter, 0))

# Per-letter speaker / FI scores
for metric in ['finger_ind_all', 'speaker_all']:
    for letter in ALPHABET:
        vals = []
        for split in SPLITS:
            if letter in scores[split][metric]:
                vals += [
                    x + 2 * y
                    for x, y in zip(scores[split][metric][letter], scores[split]['thumb'][letter])
                    if not np.isnan(x)
                ]
        data[metric].append(np.mean(remove_outliers(vals)))

# Per-pair listener (handshape distance) scores and English confusability/freq
for i, l1 in enumerate(ALPHABET):
    for l2 in ALPHABET[i + 1:]:
        bigram = l1 + l2
        data['bi_label'].append(bigram.upper())
        data['conf_word'].append(freq['conf_word'].get(bigram, 0))
        data['conf_type'].append(freq['conf_type'].get(bigram, 0))
        data['min_en_word'].append(min(freq['en_word'].get(l1, 0), freq['en_word'].get(l2, 0)))
        data['min_en_type'].append(min(freq['en_type'].get(l1, 0), freq['en_type'].get(l2, 0)))
        vals = []
        for split in SPLITS:
            if bigram in scores[split]['listener_all']:
                vals += [x for x in scores[split]['listener_all'][bigram] if not np.isnan(x)]
        data['listener_all'].append(np.mean(remove_outliers(vals)))


## Correlations (Section 5)

In [ ]:
print('--- Articulatory effort vs. handshape frequency ---')
for x in ['asl_type', 'nat_asl_type', 'for_asl_type', 'en_type']:
    corr, p = stats.spearmanr(data[x], data['finger_ind_all'])
    print(f'{x:>14s} vs finger_ind_all   r={corr:+.2f}  p={p:.3f}')

print('\n--- Perceptual effort vs. English confusability ---')
for x in ['conf_type', 'conf_word']:
    corr, p = stats.spearmanr(data[x], data['listener_all'])
    print(f'{x:>14s} vs listener_all     r={corr:+.2f}  p={p:.3f}')

print('\n--- Partial correlation, controlling for letter frequency ---')
df = pd.DataFrame({k: data[k] for k in ['conf_type', 'conf_word', 'listener_all', 'min_en_type', 'min_en_word']})
for x, z in [('conf_type', 'min_en_type'), ('conf_word', 'min_en_word')]:
    res = pg.partial_corr(data=df, x=x, y='listener_all', covar=z)
    print(f'{x:>10s} vs listener_all | {z}   r={res.r.iloc[0]:+.2f}  p={res["p-val"].iloc[0]:.3f}')


## Figure 5: scatter plots

In [ ]:
def scatter_corr(x_key, y_key, label_key, title, xlabel, ylabel, color, save_to,
                 normalize_x=True, mask_zero=True, fontsize=14):
    xs = list(data[x_key])
    ys = list(data[y_key])
    labels = list(data[label_key])
    if mask_zero:
        keep = [i for i, v in enumerate(xs) if v > 0]
        xs = [xs[i] for i in keep]
        ys = [ys[i] for i in keep]
        labels = [labels[i] for i in keep]
    if normalize_x:
        total = sum(xs)
        xs = [v / total for v in xs] if total else xs
    fig, ax = plt.subplots()
    sns.regplot(x=xs, y=ys, ci=None, scatter_kws={'s': 0}, line_kws={'color': color}, ax=ax)
    for xi, yi, lab in zip(xs, ys, labels):
        ax.text(xi, yi, lab, ha='center', va='center', fontsize=11)
    corr, pval = stats.spearmanr(xs, ys)
    ax.annotate(f'(r={corr:.2f}, p={pval:.2f})',
                xy=(0.55, 0.15), xycoords='axes fraction', fontsize=fontsize, color=color)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.savefig(save_to, bbox_inches='tight', pad_inches=0)
    plt.show()

# (a) native ASL signs vs FI
scatter_corr('nat_asl_word', 'finger_ind_all', 'char_label',
             'Core ASL signs', 'Handshape frequency', 'Finger independence',
             colors[0], f'{PLOT_DIR}/core-asl-fi.pdf')

# (b) initialized & loan signs vs FI
scatter_corr('for_asl_word', 'finger_ind_all', 'char_label',
             'Initialized & loan ASL signs', 'Handshape frequency', 'Finger independence',
             colors[3], f'{PLOT_DIR}/for-asl-fi.pdf')

# (c) English letter freq vs FI
scatter_corr('en_type', 'finger_ind_all', 'char_label',
             'ASL Fingerspelling', 'English letter frequency', 'Finger independence',
             colors[2], f'{PLOT_DIR}/en-fi.pdf', mask_zero=False)

# (d) English confusability vs handshape distance
scatter_corr('conf_type', 'listener_all', 'bi_label',
             'ASL Fingerspelling', 'English letter confusability', 'Handshape distance',
             colors[1], f'{PLOT_DIR}/en-hd.pdf', normalize_x=False, mask_zero=False)


## Sample distribution (Table 1)

In [ ]:
from collections import Counter
counts = Counter()
for split in SPLITS:
    for letter, vs in scores[split]['thumb'].items():
        counts[letter] += len(vs)
total = sum(counts.values())
for letter in sorted(counts):
    print(f'{letter.upper()} {counts[letter]:>4d} {100 * counts[letter] / total:5.1f}%')
